# NDN Multi-Model Runner (Google Colab)

This notebook runs the same project pipeline without changing your core code.

It uses:
- train_multimodel.py
- dataset/ndn_traffic.csv

Outputs are generated in:
- model/
- model_analysis/

## Step 1: Put project folder in Colab

Choose one approach:
1. Upload your full NDN folder to Colab runtime manually, or
2. Keep it in Google Drive and mount Drive below.

In [ ]:
# Optional: mount Google Drive (skip if you uploaded files directly to /content)
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# Set your project root path
# Example for Drive: /content/drive/MyDrive/NDN
# Example for uploaded folder: /content/NDN
PROJECT_ROOT = '/content/drive/MyDrive/NDN' if USE_DRIVE else '/content/NDN'
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Install required libraries for this project
!pip -q install -r "{PROJECT_ROOT}/requirements.txt"
!pip -q install xgboost seaborn scipy
print('Dependencies installed')

In [ ]:
# Validate required files
import os

required = [
    f'{PROJECT_ROOT}/train_multimodel.py',
    f'{PROJECT_ROOT}/dataset/ndn_traffic.csv'
]

for p in required:
    print(p, '->', 'OK' if os.path.exists(p) else 'MISSING')

missing = [p for p in required if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))

In [ ]:
# Run the same training script (no project code changes)
import os
import subprocess

cmd = ['python', 'train_multimodel.py']
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Training failed with exit code {result.returncode}')
print('Training complete')

In [ ]:
# Show saved metrics summary
import pickle
import pandas as pd

metrics_path = f'{PROJECT_ROOT}/model/metrics_summary.pkl'
with open(metrics_path, 'rb') as f:
    metrics = pickle.load(f)

rows = []
for name, m in metrics['individual_models'].items():
    rows.append({
        'Model': name,
        'Accuracy': m['accuracy'],
        'Precision': m['precision'],
        'Recall': m['recall'],
        'F1': m['f1'],
        'ROC_AUC': m['roc_auc'],
        'ErrorRate': m['error_rate']
    })

df = pd.DataFrame(rows).sort_values('F1', ascending=False).reset_index(drop=True)
display(df)

print('Ensemble metrics:')
display(pd.DataFrame([metrics['ensemble']]))

In [ ]:
# Display generated result images
import os
from IPython.display import Image, display, Markdown

images = [
    'confusion_matrices.png',
    'roc_curves.png',
    'model_comparison.png',
    'per_class_metrics.png',
    'neural_network_training.png',
    'error_analysis.png'
]

for name in images:
    p = f'{PROJECT_ROOT}/model_analysis/{name}'
    if os.path.exists(p):
        display(Markdown(f'### {name}'))
        display(Image(filename=p))
    else:
        print('Missing:', p)

## Run with another dataset (same project flow)

To run with a different CSV while keeping code unchanged:
1. Replace file at dataset/ndn_traffic.csv
2. Re-run the training cell

Required target column:
- Label

Recommended traffic columns:
- InInterests, OutInterests, InSatisfiedInterests, OutSatisfiedInterests
- InTimedOutInterests, OutTimedOutInterests, InNacks, OutNacks
- InData, OutData